# Clase 3 · Python profesional para LLMs y APIs

## Práctica incremental: comprender, probar y recién después modularizar

En las clases anteriores construiste un cliente mínimo, mediste tokens y latencia, diseñaste prompts verificables y validaste salidas estructuradas. En esta clase transformamos esos experimentos en una base de software mantenible.

Al terminar vas a tener:

- configuración centralizada y secretos fuera del código;
- una interfaz común para proveedores de modelos;
- clientes síncronos, asíncronos y con streaming;
- timeouts, retries con backoff y errores diferenciados;
- concurrencia controlada;
- sesiones, historial conversacional y `run_id`;
- trazas JSONL sin exponer credenciales;
- tests con dobles y mocks;
- una aplicación conversacional asíncrona reutilizable.

> **Modo seguro:** la notebook corre completa con un provider simulado. Gemini real requiere dos decisiones explícitas: `USE_REAL_GEMINI=1` en `.env` y `RUN_REAL_GEMINI_DEMO=True` en la celda opcional.

### Regla pedagógica de esta notebook

Cada componente importante sigue el mismo recorrido:

```text
problema → implementación visible → prueba → caso de borde
        → persistencia en src/ → importación → nueva verificación
```

Los módulos no se generan como bloques opacos. Primero observamos qué hacen y por qué existen; después los incorporamos al proyecto acumulativo.

## Mapa de la clase

| Bloque | Pregunta que responde | Entregable |
|---|---|---|
| Proyecto y dependencias | ¿Dónde vive cada responsabilidad? | estructura y `requirements.txt` |
| Settings y secretos | ¿Cómo cambio configuración sin editar código? | `settings.py` y `.env.example` |
| Providers | ¿Cómo evito acoplar toda la app a Gemini? | contrato común y adaptadores |
| Async y concurrencia | ¿Cómo ejecuto varias llamadas sin bloquear? | batch con semáforo |
| Resiliencia | ¿Qué hago ante timeout o rate limit? | retry selectivo con backoff |
| Streaming | ¿Cómo muestro resultados mientras llegan? | consumo incremental |
| Sesiones e historial | ¿Cómo mantengo contexto entre turnos? | `ConversationSession` |
| Trazabilidad | ¿Cómo explico qué ocurrió en una ejecución? | eventos JSONL |
| Integración y tests | ¿Cómo pruebo el conjunto sin gastar tokens? | `ConversationApp` y tests |

La idea central es separar **contrato**, **implementación**, **orquestación** y **observabilidad**.

## 1. Recuperar el proyecto incremental

**Objetivo.** Reutilizar la carpeta `ai_agent_project` creada en las clases 1 y 2.

**Qué observar.** La notebook informa qué componentes previos encuentra. No reemplaza silenciosamente los archivos de prompting o schemas creados antes.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, AsyncIterator, Callable, Literal, Protocol
from unittest.mock import AsyncMock
import asyncio
import importlib
import json
import os
import random
import sys
import time
import uuid

PROJECT_ROOT = Path.cwd() / "ai_agent_project"
SRC_DIR = PROJECT_ROOT / "src" / "ai_agent_course"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
TESTS_DIR = PROJECT_ROOT / "tests"

for directory in (SRC_DIR, ARTIFACTS_DIR, DATA_DIR, TESTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

(SRC_DIR / "__init__.py").touch()

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))


def print_json(title: str, value: Any) -> None:
    print(f"\n{title}")
    print("-" * len(title))
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print("(sin filas)")
        return
    widths = {
        column: max(len(column), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(column.ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))


print("Proyecto incremental:", PROJECT_ROOT.resolve())

Proyecto incremental: /Users/arieldelcampo/Projects/itba/daia/repos/ai-agent-developer/ai_agent_project


In [2]:
previous_components = {
    "Cliente de Clase 1": SRC_DIR / "gemini_client.py",
    "Schemas de Clase 2": SRC_DIR / "schemas.py",
    "Prompts de Clase 2": SRC_DIR / "prompting.py",
    "Confiabilidad de Clase 2": SRC_DIR / "reliability.py",
    "Reporte de Clase 1": ARTIFACTS_DIR / "class01_report.json",
    "Reporte de Clase 2": ARTIFACTS_DIR / "class02_report.json",
}

status_rows = [
    {"componente": name, "estado": "encontrado" if path.exists() else "no encontrado"}
    for name, path in previous_components.items()
]
print_table(status_rows, ["componente", "estado"])

missing = [name for name, path in previous_components.items() if not path.exists()]
if missing:
    print("\nAviso: faltan componentes previos. La notebook puede demostrar la Clase 3,")
    print("pero para la continuidad completa conviene ejecutar antes las Clases 1 y 2.")
else:
    print("\n✅ Continuidad con las Clases 1 y 2 verificada.")

componente               | estado    
-------------------------+-----------
Cliente de Clase 1       | encontrado
Schemas de Clase 2       | encontrado
Prompts de Clase 2       | encontrado
Confiabilidad de Clase 2 | encontrado
Reporte de Clase 1       | encontrado
Reporte de Clase 2       | encontrado

✅ Continuidad con las Clases 1 y 2 verificada.


## 2. Estructura del proyecto y dependencias

Una notebook sirve para experimentar y explicar. La lógica que debe reutilizar una API, un test o una clase posterior se mueve a módulos importables.

```text
ai_agent_project/
├── .env.example
├── requirements.txt
├── src/ai_agent_course/
│   ├── settings.py
│   ├── errors.py
│   ├── providers.py
│   ├── resilience.py
│   ├── conversation.py
│   ├── runtime.py
│   └── app.py
├── tests/
└── artifacts/
```

La notebook no instala paquetes automáticamente. Cambiá `RUN_INSTALLS` solo cuando el entorno realmente lo necesite.

In [4]:
RUN_INSTALLS = os.getenv("RUN_INSTALLS", "0") == "1"

REQUIREMENTS = [
    "python-dotenv>=1.0,<2.0",
    "pydantic>=2.0,<3.0",
    "pydantic-settings>=2.0,<3.0",
    "pytest>=8.0,<9.0",
    "google-genai>=1.0,<2.0",
]

requirements_path = PROJECT_ROOT / "requirements.txt"
requirements_path.write_text("\n".join(REQUIREMENTS) + "\n", encoding="utf-8")
print("✅", requirements_path.relative_to(PROJECT_ROOT))

if RUN_INSTALLS:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS])
    print("Dependencias instaladas o actualizadas.")
else:
    print("Instalación omitida.")
    print("Para instalar: python -m pip install -r ai_agent_project/requirements.txt")

✅ requirements.txt
Instalación omitida.
Para instalar: python -m pip install -r ai_agent_project/requirements.txt


## 3. Configuración y secretos

### 3.1 El problema

No queremos que el código de negocio contenga valores dispersos como API keys, modelo, timeout o concurrencia máxima. También queremos que una configuración inválida falle **al iniciar**, no después de varios minutos de ejecución.

Primero construimos `Settings` dentro de la notebook y observamos su comportamiento. Todavía no se crea ningún módulo.

In [6]:
from pydantic import Field, SecretStr, ValidationError, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """Configuración validada de la aplicación."""

    model_config = SettingsConfigDict(
        extra="ignore",
        case_sensitive=False,
        env_file_encoding="utf-8",
    )

    app_env: Literal["development", "testing", "production"] = "development"
    gemini_api_key: SecretStr | None = None
    use_real_gemini: bool = False
    default_model: str = "gemini-3.1-flash-lite"

    request_timeout_seconds: float = Field(default=20.0, gt=0)
    max_retries: int = Field(default=2, ge=0, le=8)
    retry_base_delay_seconds: float = Field(default=0.25, ge=0)
    max_concurrency: int = Field(default=3, ge=1, le=32)
    history_max_turns: int = Field(default=6, ge=1, le=50)

    input_usd_per_million: float = Field(default=0.25, ge=0)
    output_usd_per_million: float = Field(default=1.50, ge=0)

    @model_validator(mode="after")
    def validate_live_mode(self):
        """Impide activar Gemini real sin configurar una credencial."""
        if self.use_real_gemini and self.gemini_api_key is None:
            raise ValueError("USE_REAL_GEMINI=1 requiere GEMINI_API_KEY")
        return self

    @classmethod
    def from_env(cls, env_path: str | Path | None = None):
        path = Path(env_path) if env_path is not None else None
        values = {"_env_file": path} if path is not None and path.exists() else {}
        return cls(**values)

    def api_key_value(self) -> str | None:
        return self.gemini_api_key.get_secret_value() if self.gemini_api_key else None

    def safe_dict(self) -> dict:
        """Devuelve configuración apta para logs, sin revelar la API key."""
        data = self.model_dump(exclude={"gemini_api_key"})
        data["gemini_api_key"] = "***configurada***" if self.gemini_api_key else None
        return data

### 3.2 Probar antes de persistir

Vamos a comprobar cuatro propiedades:

1. el modo predeterminado no consume Gemini;
2. los límites numéricos se validan;
3. no se puede habilitar el modo real sin API key;
4. una clave configurada no aparece en salidas seguras.

In [10]:
def show_validation_case(title: str, factory: Callable[[], Settings]) -> dict:
    try:
        factory()
        return {"caso": title, "resultado": "ERROR: debía fallar"}
    except ValidationError as exc:
        first_error = exc.errors()[0]
        return {
            "caso": title,
            "resultado": "rechazado correctamente",
            "detalle": first_error["msg"],
        }


settings_demo = Settings()
settings_with_key = Settings(gemini_api_key="clave-solo-para-demostracion")

print_json("Configuración predeterminada", settings_demo.safe_dict())
print("\nRepresentación de SecretStr:", settings_with_key.gemini_api_key)
print_json("Salida segura con una key configurada", settings_with_key.safe_dict())

validation_rows = [
    show_validation_case("max_concurrency=0", lambda: Settings(max_concurrency=0)),
    show_validation_case("Gemini real sin API key", lambda: Settings(max_retries=9)),
]
print("\nCasos inválidos")
print_table(validation_rows, ["caso", "resultado", "detalle"])

assert settings_demo.use_real_gemini is False
assert settings_demo.default_model == "gemini-3.1-flash-lite"
assert settings_with_key.safe_dict()["gemini_api_key"] == "***configurada***"
assert all(row["resultado"] == "rechazado correctamente" for row in validation_rows)
print("\n✅ Settings se comporta como esperamos antes de crear settings.py.")


Configuración predeterminada
----------------------------
{
  "app_env": "development",
  "use_real_gemini": false,
  "default_model": "gemini-3.1-flash-lite",
  "request_timeout_seconds": 20.0,
  "max_retries": 2,
  "retry_base_delay_seconds": 0.25,
  "max_concurrency": 3,
  "history_max_turns": 6,
  "input_usd_per_million": 0.25,
  "output_usd_per_million": 1.5,
  "gemini_api_key": "***configurada***"
}

Representación de SecretStr: **********

Salida segura con una key configurada
-------------------------------------
{
  "app_env": "development",
  "use_real_gemini": false,
  "default_model": "gemini-3.1-flash-lite",
  "request_timeout_seconds": 20.0,
  "max_retries": 2,
  "retry_base_delay_seconds": 0.25,
  "max_concurrency": 3,
  "history_max_turns": 6,
  "input_usd_per_million": 0.25,
  "output_usd_per_million": 1.5,
  "gemini_api_key": "***configurada***"
}

Casos inválidos
caso                    | resultado               | detalle                                   
---------

### 3.3 Documentar las variables de entorno

El archivo `.env.example` se publica sin secretos. El alumno crea una copia llamada `.env` solamente si quiere configurar valores locales.

`USE_REAL_GEMINI` acepta valores booleanos habituales como `0/1` o `false/true`. En esta notebook recomendamos `0/1` para que la decisión sea visible.

In [11]:
%%writefile ai_agent_project/.env.example
# Entorno de ejecución
APP_ENV=development

# Modo seguro predeterminado: no realiza llamadas reales.
# Cambiar a 1 solamente después de configurar GEMINI_API_KEY.
USE_REAL_GEMINI=0
GEMINI_API_KEY=
DEFAULT_MODEL=gemini-3.1-flash-lite

# Resiliencia y concurrencia
REQUEST_TIMEOUT_SECONDS=20
MAX_RETRIES=2
RETRY_BASE_DELAY_SECONDS=0.25
MAX_CONCURRENCY=3
HISTORY_MAX_TURNS=6

# Valores educativos para estimación de costos.
# Revisarlos antes de usarlos para decisiones económicas reales.
INPUT_USD_PER_MILLION=0.25
OUTPUT_USD_PER_MILLION=1.50

Overwriting ai_agent_project/.env.example


### 3.4 Persistir la implementación ya probada

El siguiente bloque **no introduce una lógica nueva**. Copia al proyecto la clase que acabamos de comprender y probar, y agrega `from_env()` para cargar un archivo `.env`.

In [12]:
%%writefile ai_agent_project/src/ai_agent_course/settings.py
from __future__ import annotations

from pathlib import Path
from typing import Literal

from pydantic import Field, SecretStr, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """Configuración validada de la aplicación."""

    model_config = SettingsConfigDict(
        extra="ignore",
        case_sensitive=False,
        env_file_encoding="utf-8",
    )

    app_env: Literal["development", "testing", "production"] = "development"
    gemini_api_key: SecretStr | None = None
    use_real_gemini: bool = False
    default_model: str = "gemini-3.1-flash-lite"

    request_timeout_seconds: float = Field(default=20.0, gt=0)
    max_retries: int = Field(default=2, ge=0, le=8)
    retry_base_delay_seconds: float = Field(default=0.25, ge=0)
    max_concurrency: int = Field(default=3, ge=1, le=32)
    history_max_turns: int = Field(default=6, ge=1, le=50)

    input_usd_per_million: float = Field(default=0.25, ge=0)
    output_usd_per_million: float = Field(default=1.50, ge=0)

    @model_validator(mode="after")
    def validate_live_mode(self):
        if self.use_real_gemini and self.gemini_api_key is None:
            raise ValueError("USE_REAL_GEMINI=1 requiere GEMINI_API_KEY")
        return self

    @classmethod
    def from_env(cls, env_path: str | Path | None = None):
        path = Path(env_path) if env_path is not None else None
        values = {"_env_file": path} if path is not None and path.exists() else {}
        return cls(**values)

    def api_key_value(self) -> str | None:
        return self.gemini_api_key.get_secret_value() if self.gemini_api_key else None

    def safe_dict(self) -> dict:
        data = self.model_dump(exclude={"gemini_api_key"})
        data["gemini_api_key"] = "***configurada***" if self.gemini_api_key else None
        return data

Writing ai_agent_project/src/ai_agent_course/settings.py


In [29]:
import ai_agent_course.settings as settings_module
importlib.reload(settings_module)

settings = settings_module.Settings.from_env(PROJECT_ROOT / ".env")
demo_settings = settings_module.Settings(
    app_env="testing",
    use_real_gemini=False,
    request_timeout_seconds=0.40,
    max_retries=2,
    retry_base_delay_seconds=0.02,
    max_concurrency=3,
    history_max_turns=4,
)

print_json("Settings importado desde src/", settings.safe_dict())
assert isinstance(settings, settings_module.Settings)
assert settings.default_model == "gemini-3.1-flash-lite"
print("\n✅ El módulo importado conserva el comportamiento probado.")



Settings importado desde src/
-----------------------------
{
  "app_env": "development",
  "use_real_gemini": false,
  "default_model": "gemini-3.1-flash-lite",
  "request_timeout_seconds": 20.0,
  "max_retries": 2,
  "retry_base_delay_seconds": 0.25,
  "max_concurrency": 4,
  "history_max_turns": 6,
  "input_usd_per_million": 0.25,
  "output_usd_per_million": 1.5,
  "gemini_api_key": "***configurada***"
}

✅ El módulo importado conserva el comportamiento probado.


## 4. Errores con semántica

No todos los errores deben producir la misma política. Antes de crear `errors.py`, definimos la jerarquía y comprobamos qué fallos son transitorios.

In [14]:
class LLMError(Exception):
    """Error base de la capa de modelos."""


class ProviderConfigurationError(LLMError):
    """Configuración o credenciales inválidas. No se reintenta."""


class InvalidRequestError(LLMError):
    """Input inválido. No se reintenta."""


class InvalidProviderResponseError(LLMError):
    """El proveedor respondió, pero no cumplió el contrato."""


class TransientProviderError(LLMError):
    """Falla temporal. Puede reintentarse con límites."""


class RateLimitError(TransientProviderError):
    """Límite temporal de uso."""


class ProviderTimeoutError(TransientProviderError):
    """La llamada superó el tiempo permitido."""


class PartialStreamError(LLMError):
    """El stream falló después de emitir contenido."""


RETRYABLE_DEMO_ERRORS = (
    RateLimitError,
    ProviderTimeoutError,
    TransientProviderError,
)

error_examples = [
    ProviderConfigurationError("API key inválida"),
    InvalidRequestError("prompt vacío"),
    RateLimitError("429"),
    ProviderTimeoutError("timeout"),
    PartialStreamError("stream parcial"),
]

error_rows = [
    {
        "error": type(error).__name__,
        "retry_automático": isinstance(error, RETRYABLE_DEMO_ERRORS),
    }
    for error in error_examples
]
print_table(error_rows, ["error", "retry_automático"])

error                      | retry_automático
---------------------------+-----------------
ProviderConfigurationError | False           
InvalidRequestError        | False           
RateLimitError             | True            
ProviderTimeoutError       | True            
PartialStreamError         | False           


In [15]:
%%writefile ai_agent_project/src/ai_agent_course/errors.py
class LLMError(Exception):
    """Error base de la capa de modelos."""


class ProviderConfigurationError(LLMError):
    """Configuración o credenciales inválidas. No se reintenta."""


class InvalidRequestError(LLMError):
    """Input inválido. No se reintenta."""


class InvalidProviderResponseError(LLMError):
    """El proveedor respondió, pero no cumplió el contrato."""


class TransientProviderError(LLMError):
    """Falla temporal del proveedor. Puede reintentarse."""


class RateLimitError(TransientProviderError):
    """El proveedor rechazó temporalmente por límite de uso."""


class ProviderTimeoutError(TransientProviderError):
    """La llamada excedió el tiempo máximo permitido."""


class PartialStreamError(LLMError):
    """El stream falló después de emitir contenido; reintentar podría duplicarlo."""

Writing ai_agent_project/src/ai_agent_course/errors.py


## 5. Contrato común de providers

### 5.1 Prototipo visible

La aplicación no debería depender directamente de los objetos que devuelve cada SDK. Definimos un objeto de dominio estable (`GenerationResult`) y un contrato mínimo (`LLMProvider`).

Para probar el contrato sin red usamos `FakeProvider`, que también permite simular demoras y fallos.

In [16]:
@dataclass(frozen=True)
class GenerationResult:
    text: str
    model: str
    provider: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    finish_reason: str = "stop"

    def to_dict(self) -> dict:
        return asdict(self)


class LLMProvider(Protocol):
    def generate(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> GenerationResult:
        ...

    async def agenerate(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> GenerationResult:
        ...

    async def astream(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> AsyncIterator[str]:
        ...


class FakeProvider:
    """Doble determinístico para aprender y testear sin red ni costo."""

    def __init__(
        self,
        model: str = "fake-llm",
        *,
        delay_seconds: float = 0.03,
        failures: list[str] | None = None,
    ):
        self.model = model
        self.delay_seconds = delay_seconds
        self.failures = list(failures or [])
        self.active_calls = 0
        self.max_active_calls = 0

    def _validate(self, prompt: str) -> None:
        if not isinstance(prompt, str) or not prompt.strip():
            raise InvalidRequestError("El prompt no puede estar vacío")

    def _maybe_fail(self) -> None:
        if not self.failures:
            return
        failure = self.failures.pop(0)
        mapping = {
            "rate_limit": RateLimitError("429 simulado"),
            "transient": TransientProviderError("503 simulado"),
            "timeout": ProviderTimeoutError("timeout simulado"),
            "empty": InvalidProviderResponseError("respuesta vacía simulada"),
            "config": ProviderConfigurationError("credencial simulada inválida"),
        }
        raise mapping[failure]

    @staticmethod
    def _answer(prompt: str) -> str:
        last_line = next(
            (line.strip() for line in reversed(prompt.splitlines()) if line.strip()),
            prompt.strip(),
        )
        return f"Respuesta simulada para: {last_line[:120]}"

    def generate(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> GenerationResult:
        self._validate(prompt)
        started = time.perf_counter()
        time.sleep(self.delay_seconds)
        self._maybe_fail()
        text = self._answer(prompt)
        return GenerationResult(
            text=text,
            model=self.model,
            provider="fake",
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=max(1, len(prompt.split())),
            output_tokens=max(1, len(text.split())),
        )

    async def agenerate(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> GenerationResult:
        self._validate(prompt)
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        started = time.perf_counter()
        try:
            await asyncio.sleep(self.delay_seconds)
            self._maybe_fail()
            text = self._answer(prompt)
            return GenerationResult(
                text=text,
                model=self.model,
                provider="fake",
                latency_ms=round((time.perf_counter() - started) * 1000, 2),
                input_tokens=max(1, len(prompt.split())),
                output_tokens=max(1, len(text.split())),
            )
        finally:
            self.active_calls -= 1

    async def astream(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> AsyncIterator[str]:
        self._validate(prompt)
        self._maybe_fail()
        for word in self._answer(prompt).split():
            await asyncio.sleep(self.delay_seconds / 4)
            yield word + " "

### 5.2 Probar el contrato y sus bordes

La prueba no verifica solo que “devuelva algo”. Verifica que todas las variantes entreguen el mismo objeto de dominio y que un input inválido falle de forma explícita.

In [17]:
prototype_provider = FakeProvider(model="fake:gemini-3.1-flash-lite", delay_seconds=0.01)

sync_result = prototype_provider.generate("Explicá por qué conviene un contrato común.")
async_result = await prototype_provider.agenerate("Explicá la diferencia entre sync y async.")
stream_chunks = [chunk async for chunk in prototype_provider.astream("Mostrá una respuesta por streaming.")]

print_json("Resultado síncrono", sync_result.to_dict())
print_json("Resultado asíncrono", async_result.to_dict())
print_json("Streaming reconstruido", {"chunks": len(stream_chunks), "text": "".join(stream_chunks).strip()})

try:
    await prototype_provider.agenerate("   ")
except InvalidRequestError as exc:
    print_json("Borde validado: prompt vacío", {"error": type(exc).__name__, "message": str(exc)})

assert sync_result.provider == async_result.provider == "fake"
assert sync_result.model == async_result.model == "fake:gemini-3.1-flash-lite"
assert "".join(stream_chunks).strip()
print("\n✅ El contrato funciona antes de crear providers.py.")


Resultado síncrono
------------------
{
  "text": "Respuesta simulada para: Explicá por qué conviene un contrato común.",
  "model": "fake:gemini-3.1-flash-lite",
  "provider": "fake",
  "latency_ms": 11.22,
  "input_tokens": 7,
  "output_tokens": 10,
  "finish_reason": "stop"
}

Resultado asíncrono
-------------------
{
  "text": "Respuesta simulada para: Explicá la diferencia entre sync y async.",
  "model": "fake:gemini-3.1-flash-lite",
  "provider": "fake",
  "latency_ms": 11.09,
  "input_tokens": 7,
  "output_tokens": 10,
  "finish_reason": "stop"
}

Streaming reconstruido
----------------------
{
  "chunks": 8,
  "text": "Respuesta simulada para: Mostrá una respuesta por streaming."
}

Borde validado: prompt vacío
----------------------------
{
  "error": "InvalidRequestError",
  "message": "El prompt no puede estar vacío"
}

✅ El contrato funciona antes de crear providers.py.


### 5.3 Persistir adaptadores

El módulo conserva el `FakeProvider` probado y agrega `GeminiProvider`, que traduce el SDK oficial al mismo contrato. El resto de la aplicación no necesita conocer los tipos específicos de Google.

In [18]:
%%writefile ai_agent_project/src/ai_agent_course/providers.py
from __future__ import annotations

import asyncio
from dataclasses import asdict, dataclass
import time
from typing import AsyncIterator, Protocol

from .errors import (
    InvalidProviderResponseError,
    InvalidRequestError,
    ProviderConfigurationError,
    ProviderTimeoutError,
    RateLimitError,
    TransientProviderError,
)


@dataclass(frozen=True)
class GenerationResult:
    text: str
    model: str
    provider: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    finish_reason: str = "stop"

    def to_dict(self) -> dict:
        return asdict(self)


class LLMProvider(Protocol):
    def generate(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> GenerationResult:
        ...

    async def agenerate(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> GenerationResult:
        ...

    async def astream(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> AsyncIterator[str]:
        ...


class FakeProvider:
    """Provider determinístico para aprender y testear sin red ni costo."""

    def __init__(
        self,
        model: str = "fake-llm",
        *,
        delay_seconds: float = 0.03,
        failures: list[str] | None = None,
    ):
        self.model = model
        self.delay_seconds = delay_seconds
        self.failures = list(failures or [])
        self.active_calls = 0
        self.max_active_calls = 0

    def _validate(self, prompt: str) -> None:
        if not isinstance(prompt, str) or not prompt.strip():
            raise InvalidRequestError("El prompt no puede estar vacío")

    def _maybe_fail(self) -> None:
        if not self.failures:
            return
        failure = self.failures.pop(0)
        mapping = {
            "rate_limit": RateLimitError("429 simulado: demasiadas solicitudes"),
            "transient": TransientProviderError("503 simulado: proveedor no disponible"),
            "timeout": ProviderTimeoutError("timeout simulado por el provider"),
            "empty": InvalidProviderResponseError("respuesta simulada vacía"),
            "config": ProviderConfigurationError("credencial simulada inválida"),
        }
        raise mapping[failure]

    @staticmethod
    def _answer(prompt: str) -> str:
        marker = "NUEVO MENSAJE DEL USUARIO"
        if marker in prompt:
            tail = prompt.split(marker, 1)[1]
            user_text = tail.split("Respondé considerando el historial reciente.", 1)[0].strip()
            return f"Respuesta simulada para: {user_text[:120]}"
        last_line = next(
            (line.strip() for line in reversed(prompt.splitlines()) if line.strip()),
            prompt.strip(),
        )
        return f"Respuesta simulada para: {last_line[:120]}"

    def generate(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> GenerationResult:
        self._validate(prompt)
        started = time.perf_counter()
        time.sleep(self.delay_seconds)
        self._maybe_fail()
        text = self._answer(prompt)
        return GenerationResult(
            text=text,
            model=self.model,
            provider="fake",
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=max(1, len(prompt.split())),
            output_tokens=max(1, len(text.split())),
        )

    async def agenerate(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> GenerationResult:
        self._validate(prompt)
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        started = time.perf_counter()
        try:
            await asyncio.sleep(self.delay_seconds)
            self._maybe_fail()
            text = self._answer(prompt)
            return GenerationResult(
                text=text,
                model=self.model,
                provider="fake",
                latency_ms=round((time.perf_counter() - started) * 1000, 2),
                input_tokens=max(1, len(prompt.split())),
                output_tokens=max(1, len(text.split())),
            )
        finally:
            self.active_calls -= 1

    async def astream(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> AsyncIterator[str]:
        self._validate(prompt)
        self._maybe_fail()
        for word in self._answer(prompt).split():
            await asyncio.sleep(self.delay_seconds / 4)
            yield word + " "


class GeminiProvider:
    """Adaptador opcional para Google Gen AI SDK."""

    def __init__(self, api_key: str | None, model: str):
        if not api_key:
            raise ProviderConfigurationError("Falta GEMINI_API_KEY")
        from google import genai
        self.model = model
        self.client = genai.Client(api_key=api_key)

    @staticmethod
    def _map_exception(exc: Exception) -> Exception:
        text = str(exc).lower()
        if "429" in text or "rate limit" in text or "resource exhausted" in text:
            return RateLimitError(str(exc))
        if "timeout" in text or "timed out" in text:
            return ProviderTimeoutError(str(exc))
        if "401" in text or "403" in text or "api key" in text:
            return ProviderConfigurationError(str(exc))
        return TransientProviderError(str(exc))

    @staticmethod
    def _config(system: str | None, temperature: float):
        from google.genai import types
        return types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
        )

    @staticmethod
    def _usage(response) -> tuple[int, int]:
        usage = getattr(response, "usage_metadata", None)
        return (
            int(getattr(usage, "prompt_token_count", 0) or 0),
            int(getattr(usage, "candidates_token_count", 0) or 0),
        )

    def _normalize(self, response, started: float) -> GenerationResult:
        text = response.text or ""
        if not text.strip():
            raise InvalidProviderResponseError("Gemini devolvió texto vacío")
        input_tokens, output_tokens = self._usage(response)
        return GenerationResult(
            text=text,
            model=self.model,
            provider="gemini",
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=input_tokens,
            output_tokens=output_tokens,
        )

    def generate(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> GenerationResult:
        if not prompt.strip():
            raise InvalidRequestError("El prompt no puede estar vacío")
        started = time.perf_counter()
        try:
            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config=self._config(system, temperature),
            )
            return self._normalize(response, started)
        except Exception as exc:
            if isinstance(exc, InvalidProviderResponseError):
                raise
            raise self._map_exception(exc) from exc

    async def agenerate(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> GenerationResult:
        if not prompt.strip():
            raise InvalidRequestError("El prompt no puede estar vacío")
        started = time.perf_counter()
        try:
            response = await self.client.aio.models.generate_content(
                model=self.model,
                contents=prompt,
                config=self._config(system, temperature),
            )
            return self._normalize(response, started)
        except Exception as exc:
            if isinstance(exc, InvalidProviderResponseError):
                raise
            raise self._map_exception(exc) from exc

    async def astream(self, prompt: str, *, system: str | None = None, temperature: float = 0.2) -> AsyncIterator[str]:
        if not prompt.strip():
            raise InvalidRequestError("El prompt no puede estar vacío")
        try:
            stream = await self.client.aio.models.generate_content_stream(
                model=self.model,
                contents=prompt,
                config=self._config(system, temperature),
            )
            async for chunk in stream:
                text = chunk.text or ""
                if text:
                    yield text
        except Exception as exc:
            raise self._map_exception(exc) from exc


def build_provider(settings):
    if settings.use_real_gemini:
        return GeminiProvider(settings.api_key_value(), settings.default_model)
    return FakeProvider(model=f"fake:{settings.default_model}")

Writing ai_agent_project/src/ai_agent_course/providers.py


In [20]:
import ai_agent_course.providers as providers_module
importlib.reload(providers_module)

provider = providers_module.build_provider(settings) #si queremos usar tus settings definidas remplazamos demo_settings por settings
module_result = provider.generate(
    "Explicá en una oración por qué conviene abstraer el proveedor.",
    system="Respondé de forma breve.",
)

print_json("Resultado desde providers.py", module_result.to_dict())
# si usas setting descomentar la siguente linea y comnetar la proxima
#assert module_result.provider == ("gemini" if settings.use_real_gemini else "fake")
assert module_result.provider ==  "fake"
#assert module_result.model == "gemini-3.1-flash-lite"
assert module_result.model == "fake:gemini-3.1-flash-lite"
print("\n✅ El módulo persistido respeta el contrato probado.")


Resultado desde providers.py
----------------------------
{
  "text": "Respuesta simulada para: Explicá en una oración por qué conviene abstraer el proveedor.",
  "model": "fake:gemini-3.1-flash-lite",
  "provider": "fake",
  "latency_ms": 33.65,
  "input_tokens": 10,
  "output_tokens": 13,
  "finish_reason": "stop"
}

✅ El módulo persistido respeta el contrato probado.


## 6. Síncrono, asíncrono y concurrencia controlada

- `generate()` bloquea hasta terminar.
- `agenerate()` cede el control mientras espera I/O.
- `asyncio.gather()` coordina tareas concurrentes.
- `Semaphore` evita una cantidad ilimitada de llamadas simultáneas.

El resultado funcional es equivalente; lo que cambia es el uso del tiempo y la presión sobre el proveedor.

In [30]:
async def run_sequential(prompts: list[str], provider) -> list:
    results = []
    for prompt in prompts:
        results.append(await provider.agenerate(prompt))
    return results


async def run_concurrent(prompts: list[str], provider, *, max_concurrency: int) -> list:
    semaphore = asyncio.Semaphore(max_concurrency)

    async def one(index: int, prompt: str):
        async with semaphore:
            result = await provider.agenerate(prompt)
            return index, result

    indexed = await asyncio.gather(
        *(one(index, prompt) for index, prompt in enumerate(prompts))
    )
    return [result for _, result in sorted(indexed)]


prompts = [f"Solicitud interna número {index}" for index in range(1, 7)]

sequential_provider = providers_module.FakeProvider(delay_seconds=0.08)
started = time.perf_counter()
sequential_results = await run_sequential(prompts, sequential_provider)
sequential_ms = round((time.perf_counter() - started) * 1000, 1)

concurrent_provider = providers_module.FakeProvider(delay_seconds=0.08)
started = time.perf_counter()
concurrent_results = await run_concurrent(
    prompts,
    concurrent_provider,
    max_concurrency=demo_settings.max_concurrency,
)
concurrent_ms = round((time.perf_counter() - started) * 1000, 1)

comparison_rows = [
    {
        "modo": "secuencial",
        "llamadas": len(sequential_results),
        "tiempo_ms": sequential_ms,
        "máx_simultáneas": sequential_provider.max_active_calls,
    },
    {
        "modo": "concurrente",
        "llamadas": len(concurrent_results),
        "tiempo_ms": concurrent_ms,
        "máx_simultáneas": concurrent_provider.max_active_calls,
    },
]
print_table(comparison_rows, ["modo", "llamadas", "tiempo_ms", "máx_simultáneas"])
print(f"\nAceleración observada: {sequential_ms / concurrent_ms:.2f}x")

modo        | llamadas | tiempo_ms | máx_simultáneas
------------+----------+-----------+----------------
secuencial  | 6        | 486.3     | 1              
concurrente | 6        | 161.9     | 3              

Aceleración observada: 3.00x


### Actividad 1 · Elegir un límite

Probá `max_concurrency=1`, `2`, `3` y `6`. Registrá qué cambia y qué riesgos tendría usar concurrencia ilimitada con un proveedor real.

> Más concurrencia no siempre es mejor: puede aumentar rate limits, consumo simultáneo y dificultad de diagnóstico.

## 7. Timeouts, retries y backoff

Primero implementamos la política dentro de la notebook. La función reintenta solamente errores transitorios, impone un timeout y devuelve evidencia de cada intento.

In [31]:
RETRYABLE_ERRORS = (
    providers_module.RateLimitError,
    providers_module.ProviderTimeoutError,
    providers_module.TransientProviderError,
)


@dataclass(frozen=True)
class RetryEvent:
    attempt: int
    error_type: str
    message: str
    next_delay_seconds: float


@dataclass(frozen=True)
class CallOutcome:
    result: providers_module.GenerationResult
    attempts: int
    retries: tuple[RetryEvent, ...]


async def generate_with_resilience(
    provider: providers_module.LLMProvider,
    prompt: str,
    *,
    system: str | None = None,
    temperature: float = 0.2,
    timeout_seconds: float = 20.0,
    max_retries: int = 2,
    base_delay_seconds: float = 0.25,
    jitter: bool = False,
    on_retry: Callable[[RetryEvent], None] | None = None,
) -> CallOutcome:
    retries: list[RetryEvent] = []

    for attempt in range(1, max_retries + 2):
        try:
            result = await asyncio.wait_for(
                provider.agenerate(prompt, system=system, temperature=temperature),
                timeout=timeout_seconds,
            )
            return CallOutcome(result=result, attempts=attempt, retries=tuple(retries))
        except asyncio.TimeoutError:
            error = providers_module.ProviderTimeoutError(
                f"La llamada superó {timeout_seconds:.2f} segundos"
            )
        except RETRYABLE_ERRORS as exc:
            error = exc

        if attempt > max_retries:
            raise error

        delay = base_delay_seconds * (2 ** (attempt - 1))
        if jitter:
            delay *= random.uniform(0.8, 1.2)

        event = RetryEvent(
            attempt=attempt,
            error_type=type(error).__name__,
            message=str(error),
            next_delay_seconds=round(delay, 4),
        )
        retries.append(event)
        if on_retry:
            on_retry(event)
        await asyncio.sleep(delay)

    raise RuntimeError("Estado inalcanzable")

In [32]:
retry_log: list[RetryEvent] = []
flaky_provider = providers_module.FakeProvider(
    delay_seconds=0.01,
    failures=["rate_limit"],
)


prototype_outcome = await generate_with_resilience(
    flaky_provider,
    "Procesá esta solicitud después de un rate limit transitorio.",
    timeout_seconds=0.30,
    max_retries=2,
    base_delay_seconds=0.01,
    on_retry=retry_log.append,
)

print_table(
    [
        {
            "intento_fallido": event.attempt,
            "error": event.error_type,
            "espera_s": event.next_delay_seconds,
        }
        for event in retry_log
    ],
    ["intento_fallido", "error", "espera_s"],
)
print_json("Resultado final", {
    "attempts": prototype_outcome.attempts,
    "text": prototype_outcome.result.text,
})

non_retryable = providers_module.FakeProvider(delay_seconds=0, failures=["config"])
try:
    await generate_with_resilience(non_retryable, "mensaje", max_retries=3)
except providers_module.ProviderConfigurationError as exc:
    print_json("Error no reintentable", {"type": type(exc).__name__, "message": str(exc)})

assert prototype_outcome.attempts == 2
assert len(prototype_outcome.retries) == 1
print("\n✅ La política se verificó antes de crear resilience.py.")

intento_fallido | error          | espera_s
----------------+----------------+---------
1               | RateLimitError | 0.01    

Resultado final
---------------
{
  "attempts": 2,
  "text": "Respuesta simulada para: Procesá esta solicitud después de un rate limit transitorio."
}

Error no reintentable
---------------------
{
  "type": "ProviderConfigurationError",
  "message": "credencial simulada inválida"
}

✅ La política se verificó antes de crear resilience.py.


In [33]:
%%writefile ai_agent_project/src/ai_agent_course/resilience.py
from __future__ import annotations

from dataclasses import dataclass
import asyncio
import random
from typing import Callable

from .errors import ProviderTimeoutError, RateLimitError, TransientProviderError
from .providers import GenerationResult, LLMProvider


RETRYABLE_ERRORS = (RateLimitError, ProviderTimeoutError, TransientProviderError)


@dataclass(frozen=True)
class RetryEvent:
    attempt: int
    error_type: str
    message: str
    next_delay_seconds: float


@dataclass(frozen=True)
class CallOutcome:
    result: GenerationResult
    attempts: int
    retries: tuple[RetryEvent, ...]


async def generate_with_resilience(
    provider: LLMProvider,
    prompt: str,
    *,
    system: str | None = None,
    temperature: float = 0.2,
    timeout_seconds: float = 20.0,
    max_retries: int = 2,
    base_delay_seconds: float = 0.25,
    jitter: bool = False,
    on_retry: Callable[[RetryEvent], None] | None = None,
) -> CallOutcome:
    retries: list[RetryEvent] = []

    for attempt in range(1, max_retries + 2):
        try:
            result = await asyncio.wait_for(
                provider.agenerate(prompt, system=system, temperature=temperature),
                timeout=timeout_seconds,
            )
            return CallOutcome(result=result, attempts=attempt, retries=tuple(retries))
        except asyncio.TimeoutError:
            error = ProviderTimeoutError(
                f"La llamada superó {timeout_seconds:.2f} segundos"
            )
        except RETRYABLE_ERRORS as exc:
            error = exc

        if attempt > max_retries:
            raise error

        delay = base_delay_seconds * (2 ** (attempt - 1))
        if jitter:
            delay *= random.uniform(0.8, 1.2)

        event = RetryEvent(
            attempt=attempt,
            error_type=type(error).__name__,
            message=str(error),
            next_delay_seconds=round(delay, 4),
        )
        retries.append(event)
        if on_retry:
            on_retry(event)
        await asyncio.sleep(delay)

    raise RuntimeError("Estado inalcanzable")

Writing ai_agent_project/src/ai_agent_course/resilience.py


In [34]:
import ai_agent_course.resilience as resilience_module
importlib.reload(resilience_module)

module_outcome = await resilience_module.generate_with_resilience(
    providers_module.FakeProvider(delay_seconds=0.01, failures=["rate_limit"]),
    "Prueba del módulo persistido.",
    timeout_seconds=0.3,
    max_retries=1,
    base_delay_seconds=0,
)
print_json("Resultado desde resilience.py", {
    "attempts": module_outcome.attempts,
    "retries": [asdict(event) for event in module_outcome.retries],
})
assert module_outcome.attempts == 2


Resultado desde resilience.py
-----------------------------
{
  "attempts": 2,
  "retries": [
    {
      "attempt": 1,
      "error_type": "RateLimitError",
      "message": "429 simulado: demasiadas solicitudes",
      "next_delay_seconds": 0
    }
  ]
}


## 8. Streaming

El streaming reduce el tiempo percibido hasta el primer fragmento. También introduce una decisión importante: si el stream falla después de mostrar texto, un retry automático podría duplicar contenido.

In [35]:
async def capture_stream(provider, prompt: str) -> tuple[str, list[dict[str, Any]]]:
    started = time.perf_counter()
    chunks: list[str] = []
    events: list[dict[str, Any]] = []

    async for chunk in provider.astream(prompt):
        chunks.append(chunk)
        events.append({
            "chunk": len(chunks),
            "elapsed_ms": round((time.perf_counter() - started) * 1000, 1),
            "text": chunk.strip(),
        })

    return "".join(chunks).strip(), events


stream_provider = providers_module.FakeProvider(delay_seconds=0.04)
stream_text, stream_events = await capture_stream(
    stream_provider,
    "Resumí por qué el streaming mejora la experiencia.",
)

print("Primeros fragmentos:")
print_table(stream_events[:6], ["chunk", "elapsed_ms", "text"])
print_json("Respuesta reconstruida", {
    "chunks": len(stream_events),
    "text": stream_text,
    "first_chunk_ms": stream_events[0]["elapsed_ms"],
    "total_ms": stream_events[-1]["elapsed_ms"],
})

Primeros fragmentos:
chunk | elapsed_ms | text     
------+------------+----------
1     | 11.1       | Respuesta
2     | 22.0       | simulada 
3     | 33.1       | para:    
4     | 44.1       | Resumí   
5     | 55.2       | por      
6     | 66.3       | qué      

Respuesta reconstruida
----------------------
{
  "chunks": 11,
  "text": "Respuesta simulada para: Resumí por qué el streaming mejora la experiencia.",
  "first_chunk_ms": 11.1,
  "total_ms": 120.7
}


### Gemini real opcional: dos controles explícitos

La notebook utiliza dos controles para evitar consumos accidentales:

1. En `ai_agent_project/.env`, configurá `GEMINI_API_KEY` y cambiá `USE_REAL_GEMINI=0` por `USE_REAL_GEMINI=1`.
2. En la siguiente celda, cambiá `RUN_REAL_GEMINI_DEMO = False` por `True`.

Si alguno permanece desactivado, la llamada real no se ejecutará. Nunca pegues la API key directamente en la notebook.

In [36]:
# Segunda confirmación de seguridad.
# Cambiar a True únicamente cuando:
# 1. GEMINI_API_KEY esté configurada en ai_agent_project/.env
# 2. USE_REAL_GEMINI=1 esté configurado en ese mismo archivo.
RUN_REAL_GEMINI_DEMO = True

if RUN_REAL_GEMINI_DEMO:
    real_settings = settings_module.Settings.from_env(PROJECT_ROOT / ".env")

    if not real_settings.use_real_gemini:
        raise RuntimeError(
            "RUN_REAL_GEMINI_DEMO=True, pero USE_REAL_GEMINI sigue desactivado en .env"
        )

    real_provider = providers_module.build_provider(real_settings)
    real_outcome = await resilience_module.generate_with_resilience(
        real_provider,
        "Explicá en dos oraciones la diferencia entre async y streaming.",
        system="Sos un asistente técnico y conciso.",
        timeout_seconds=real_settings.request_timeout_seconds,
        max_retries=real_settings.max_retries,
        base_delay_seconds=real_settings.retry_base_delay_seconds,
    )
    print_json("Respuesta real", real_outcome.result.to_dict())
else:
    print("Gemini real no fue ejecutado.")
    print("Para activarlo: configurá .env y cambiá RUN_REAL_GEMINI_DEMO a True.")


Respuesta real
--------------
{
  "text": "`Async` permite que una aplicación realice otras tareas mientras espera la respuesta completa de una operación, evitando bloqueos. El `streaming` procesa y entrega los datos por partes a medida que llegan, permitiendo visualizar o utilizar la información antes de que la transferencia total finalice.",
  "model": "gemini-3.1-flash-lite",
  "provider": "gemini",
  "latency_ms": 1537.88,
  "input_tokens": 24,
  "output_tokens": 56,
  "finish_reason": "stop"
}


## 9. Sesiones e historial conversacional

Una sesión identifica una conversación; un `run_id` identifica una ejecución dentro de esa sesión. El historial conserva roles y aplica una ventana explícita para no crecer indefinidamente.

Primero probamos el modelo de sesión dentro de la notebook.

In [37]:
@dataclass(frozen=True)
class Message:
    role: str
    content: str
    created_at: str

    @classmethod
    def create(cls, role: str, content: str) -> "Message":
        if role not in {"user", "assistant"}:
            raise ValueError("role debe ser user o assistant")
        if not content.strip():
            raise ValueError("content no puede estar vacío")
        return cls(
            role=role,
            content=content.strip(),
            created_at=datetime.now(timezone.utc).isoformat(),
        )

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class ConversationSession:
    session_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    max_turns: int = 6
    messages: list[Message] = field(default_factory=list)

    def add(self, role: str, content: str) -> Message:
        message = Message.create(role, content)
        self.messages.append(message)
        return message

    def recent_messages(self) -> list[Message]:
        return self.messages[-self.max_turns * 2:]

    def build_prompt(self, user_text: str) -> str:
        if not user_text.strip():
            raise ValueError("user_text no puede estar vacío")
        history = "\n".join(
            f"{message.role.upper()}: {message.content}"
            for message in self.recent_messages()
        ) or "(sin historial previo)"
        return f"""
HISTORIAL RECIENTE
{history}

NUEVO MENSAJE DEL USUARIO
{user_text.strip()}

Respondé considerando el historial reciente.
""".strip()

    def transcript(self) -> list[dict]:
        return [message.to_dict() for message in self.messages]

In [38]:
prototype_session = ConversationSession(max_turns=2)
prototype_session.add("user", "Mi nombre es Ana.")
prototype_session.add("assistant", "Hola Ana, ¿en qué te ayudo?")
prototype_session.add("user", "Estoy armando un cliente asíncrono.")
prototype_session.add("assistant", "Podés separar provider y orquestación.")

prompt_with_history = prototype_session.build_prompt("¿Qué estaba armando?")
print(prompt_with_history)
print(f"\nMensajes almacenados: {len(prototype_session.messages)}")
print(f"Mensajes enviados en la ventana: {len(prototype_session.recent_messages())}")

try:
    prototype_session.add("system", "rol no permitido")
except ValueError as exc:
    print_json("Borde validado", {"error": type(exc).__name__, "message": str(exc)})

assert len(prototype_session.recent_messages()) == 4
print("\n✅ La sesión se verificó antes de crear conversation.py.")

HISTORIAL RECIENTE
USER: Mi nombre es Ana.
ASSISTANT: Hola Ana, ¿en qué te ayudo?
USER: Estoy armando un cliente asíncrono.
ASSISTANT: Podés separar provider y orquestación.

NUEVO MENSAJE DEL USUARIO
¿Qué estaba armando?

Respondé considerando el historial reciente.

Mensajes almacenados: 4
Mensajes enviados en la ventana: 4

Borde validado
--------------
{
  "error": "ValueError",
  "message": "role debe ser user o assistant"
}

✅ La sesión se verificó antes de crear conversation.py.


In [39]:
%%writefile ai_agent_project/src/ai_agent_course/conversation.py
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
import uuid


@dataclass(frozen=True)
class Message:
    role: str
    content: str
    created_at: str

    @classmethod
    def create(cls, role: str, content: str) -> "Message":
        if role not in {"user", "assistant"}:
            raise ValueError("role debe ser user o assistant")
        if not content.strip():
            raise ValueError("content no puede estar vacío")
        return cls(
            role=role,
            content=content.strip(),
            created_at=datetime.now(timezone.utc).isoformat(),
        )

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class ConversationSession:
    session_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    max_turns: int = 6
    messages: list[Message] = field(default_factory=list)

    def add(self, role: str, content: str) -> Message:
        message = Message.create(role, content)
        self.messages.append(message)
        return message

    def recent_messages(self) -> list[Message]:
        return self.messages[-self.max_turns * 2:]

    def build_prompt(self, user_text: str) -> str:
        if not user_text.strip():
            raise ValueError("user_text no puede estar vacío")
        history = "\n".join(
            f"{message.role.upper()}: {message.content}"
            for message in self.recent_messages()
        ) or "(sin historial previo)"
        return f"""
HISTORIAL RECIENTE
{history}

NUEVO MENSAJE DEL USUARIO
{user_text.strip()}

Respondé considerando el historial reciente.
""".strip()

    def transcript(self) -> list[dict]:
        return [message.to_dict() for message in self.messages]

Writing ai_agent_project/src/ai_agent_course/conversation.py


In [40]:
import ai_agent_course.conversation as conversation_module
importlib.reload(conversation_module)

session = conversation_module.ConversationSession(max_turns=2)
session.add("user", "Mi nombre es Ana.")
session.add("assistant", "Hola Ana, ¿en qué te ayudo?")
module_prompt = session.build_prompt("¿Recordás mi nombre?")
print(module_prompt)
assert "Mi nombre es Ana" in module_prompt
print("\n✅ conversation.py importado y verificado.")

HISTORIAL RECIENTE
USER: Mi nombre es Ana.
ASSISTANT: Hola Ana, ¿en qué te ayudo?

NUEVO MENSAJE DEL USUARIO
¿Recordás mi nombre?

Respondé considerando el historial reciente.

✅ conversation.py importado y verificado.


## 10. Trazabilidad por sesión y ejecución

Una traza útil permite reconstruir qué pasó sin depender de `print()` dispersos. En esta clase usamos JSONL porque es fácil de inspeccionar y cada evento se agrega de forma independiente.

In [41]:
@dataclass(frozen=True)
class TraceEvent:
    event_id: str
    run_id: str
    session_id: str
    event: str
    timestamp: str
    payload: dict


class JsonlTraceStore:
    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def append(self, *, run_id: str, session_id: str, event: str, payload: dict) -> TraceEvent:
        item = TraceEvent(
            event_id=str(uuid.uuid4()),
            run_id=run_id,
            session_id=session_id,
            event=event,
            timestamp=datetime.now(timezone.utc).isoformat(),
            payload=payload,
        )
        with self.path.open("a", encoding="utf-8") as file:
            file.write(json.dumps(asdict(item), ensure_ascii=False, default=str) + "\n")
        return item

    def read_all(self) -> list[dict]:
        if not self.path.exists():
            return []
        return [
            json.loads(line)
            for line in self.path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

In [42]:
prototype_trace_path = ARTIFACTS_DIR / "class03_trace_prototype.jsonl"
if prototype_trace_path.exists():
    prototype_trace_path.unlink()

prototype_store = JsonlTraceStore(prototype_trace_path)
prototype_store.append(
    run_id="run-demo",
    session_id="session-demo",
    event="run.started",
    payload={"provider": "fake", "api_key": "NO DEBE REGISTRARSE"},
)

prototype_events = prototype_store.read_all()
# Una traza no debería recibir secretos. Eliminamos el archivo de prueba y repetimos correctamente.
prototype_trace_path.unlink()
prototype_store.append(
    run_id="run-demo",
    session_id="session-demo",
    event="run.started",
    payload={"provider": "fake", "model": "fake:gemini-3.1-flash-lite"},
)
prototype_events = prototype_store.read_all()

print_json("Evento persistido", prototype_events[0])
assert "api_key" not in prototype_events[0]["payload"]
print("\n✅ La traza se probó sin almacenar secretos.")


Evento persistido
-----------------
{
  "event_id": "c79beea7-e758-4d75-ada4-0c129965702a",
  "run_id": "run-demo",
  "session_id": "session-demo",
  "event": "run.started",
  "timestamp": "2026-08-08T20:31:54.378484+00:00",
  "payload": {
    "provider": "fake",
    "model": "fake:gemini-3.1-flash-lite"
  }
}

✅ La traza se probó sin almacenar secretos.


In [43]:
%%writefile ai_agent_project/src/ai_agent_course/runtime.py
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import json
from pathlib import Path
import uuid


@dataclass(frozen=True)
class TraceEvent:
    event_id: str
    run_id: str
    session_id: str
    event: str
    timestamp: str
    payload: dict


class JsonlTraceStore:
    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def append(self, *, run_id: str, session_id: str, event: str, payload: dict) -> TraceEvent:
        item = TraceEvent(
            event_id=str(uuid.uuid4()),
            run_id=run_id,
            session_id=session_id,
            event=event,
            timestamp=datetime.now(timezone.utc).isoformat(),
            payload=payload,
        )
        with self.path.open("a", encoding="utf-8") as file:
            file.write(json.dumps(asdict(item), ensure_ascii=False, default=str) + "\n")
        return item

    def read_all(self) -> list[dict]:
        if not self.path.exists():
            return []
        return [
            json.loads(line)
            for line in self.path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

Writing ai_agent_project/src/ai_agent_course/runtime.py


## 11. Caso integrador · Orquestar sin acoplar

Antes de crear `app.py`, armamos una versión visible de la orquestación. La aplicación:

1. recibe sesión y mensaje;
2. construye el prompt con historial;
3. crea un `run_id`;
4. ejecuta el provider con timeout y retry;
5. actualiza el historial solo si la respuesta termina bien;
6. calcula costo estimado;
7. registra eventos;
8. devuelve un objeto de dominio a la interfaz.

In [44]:
@dataclass(frozen=True)
class ChatResponsePrototype:
    run_id: str
    session_id: str
    text: str
    attempts: int
    provider: str
    model: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    estimated_cost_usd: float

    def to_dict(self) -> dict:
        return asdict(self)


class ConversationAppPrototype:
    def __init__(self, provider, settings, trace_store):
        self.provider = provider
        self.settings = settings
        self.trace_store = trace_store

    def _estimate_cost(self, input_tokens: int, output_tokens: int) -> float:
        return round(
            input_tokens / 1_000_000 * self.settings.input_usd_per_million
            + output_tokens / 1_000_000 * self.settings.output_usd_per_million,
            8,
        )

    async def ask(self, session, user_text: str, *, system: str = "Sos un asistente útil y breve."):
        clean_text = user_text.strip()
        if not clean_text:
            raise ValueError("El mensaje no puede estar vacío")

        run_id = str(uuid.uuid4())
        prompt = session.build_prompt(clean_text)
        self.trace_store.append(
            run_id=run_id,
            session_id=session.session_id,
            event="run.started",
            payload={"provider": type(self.provider).__name__},
        )

        outcome = await resilience_module.generate_with_resilience(
            self.provider,
            prompt,
            system=system,
            timeout_seconds=self.settings.request_timeout_seconds,
            max_retries=self.settings.max_retries,
            base_delay_seconds=self.settings.retry_base_delay_seconds,
        )

        session.add("user", clean_text)
        session.add("assistant", outcome.result.text)

        response = ChatResponsePrototype(
            run_id=run_id,
            session_id=session.session_id,
            text=outcome.result.text,
            attempts=outcome.attempts,
            provider=outcome.result.provider,
            model=outcome.result.model,
            latency_ms=outcome.result.latency_ms,
            input_tokens=outcome.result.input_tokens,
            output_tokens=outcome.result.output_tokens,
            estimated_cost_usd=self._estimate_cost(
                outcome.result.input_tokens,
                outcome.result.output_tokens,
            ),
        )
        self.trace_store.append(
            run_id=run_id,
            session_id=session.session_id,
            event="run.completed",
            payload=response.to_dict(),
        )
        return response

In [45]:
prototype_app_trace = ARTIFACTS_DIR / "class03_app_prototype.jsonl"
if prototype_app_trace.exists():
    prototype_app_trace.unlink()

prototype_app = ConversationAppPrototype(
    providers_module.FakeProvider(delay_seconds=0.01, failures=["rate_limit"]),
    demo_settings,
    JsonlTraceStore(prototype_app_trace),
)
prototype_app_session = conversation_module.ConversationSession(max_turns=2)
prototype_response = await prototype_app.ask(
    prototype_app_session,
    "¿Qué responsabilidad debería tener un provider?",
)

print_json("Respuesta del prototipo integrado", prototype_response.to_dict())
print_json("Eventos del prototipo", JsonlTraceStore(prototype_app_trace).read_all())
assert prototype_response.attempts == 2
assert len(prototype_app_session.messages) == 2
print("\n✅ La orquestación se verificó antes de crear app.py.")


Respuesta del prototipo integrado
---------------------------------
{
  "run_id": "f6028a8c-f2fb-43d1-a07c-ab23d0f2ac3f",
  "session_id": "ef850837-881f-46f2-8951-c1cd2c0ec97e",
  "text": "Respuesta simulada para: ¿Qué responsabilidad debería tener un provider?",
  "attempts": 2,
  "provider": "fake",
  "model": "fake-llm",
  "latency_ms": 11.06,
  "input_tokens": 20,
  "output_tokens": 9,
  "estimated_cost_usd": 1.85e-05
}

Eventos del prototipo
---------------------
[
  {
    "event_id": "1136cd19-78e0-4e54-9ad5-42ec7e0c7b2a",
    "run_id": "f6028a8c-f2fb-43d1-a07c-ab23d0f2ac3f",
    "session_id": "ef850837-881f-46f2-8951-c1cd2c0ec97e",
    "event": "run.started",
    "timestamp": "2026-08-08T20:33:54.870367+00:00",
    "payload": {
      "provider": "FakeProvider"
    }
  },
  {
    "event_id": "ccd5e3e3-f4bb-479c-9953-64728465d3b1",
    "run_id": "f6028a8c-f2fb-43d1-a07c-ab23d0f2ac3f",
    "session_id": "ef850837-881f-46f2-8951-c1cd2c0ec97e",
    "event": "run.completed",
    "tim

### Persistir la aplicación

El módulo agrega también `ask_stream()`. Esa variante mide tiempo al primer fragmento y evita reintentar silenciosamente un stream parcial.

In [46]:
%%writefile ai_agent_project/src/ai_agent_course/app.py
from __future__ import annotations

from dataclasses import asdict, dataclass
import time
import uuid

from .conversation import ConversationSession
from .errors import PartialStreamError
from .resilience import RetryEvent, generate_with_resilience
from .runtime import JsonlTraceStore


@dataclass(frozen=True)
class ChatResponse:
    run_id: str
    session_id: str
    text: str
    attempts: int
    provider: str
    model: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    estimated_cost_usd: float

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass(frozen=True)
class StreamChatResponse:
    run_id: str
    session_id: str
    text: str
    chunks: int
    first_chunk_ms: float
    total_ms: float
    provider: str
    model: str

    def to_dict(self) -> dict:
        return asdict(self)


class ConversationApp:
    def __init__(self, provider, settings, trace_store: JsonlTraceStore):
        self.provider = provider
        self.settings = settings
        self.trace_store = trace_store

    def _estimate_cost(self, input_tokens: int, output_tokens: int) -> float:
        return round(
            input_tokens / 1_000_000 * self.settings.input_usd_per_million
            + output_tokens / 1_000_000 * self.settings.output_usd_per_million,
            8,
        )

    def _start_run(self, *, run_id: str, session: ConversationSession, mode: str) -> None:
        self.trace_store.append(
            run_id=run_id,
            session_id=session.session_id,
            event="run.started",
            payload={
                "mode": mode,
                "provider": type(self.provider).__name__,
                "model": getattr(self.provider, "model", "unknown"),
                "history_messages": len(session.recent_messages()),
            },
        )

    async def ask(
        self,
        session: ConversationSession,
        user_text: str,
        *,
        system: str = "Sos un asistente útil, preciso y breve.",
    ) -> ChatResponse:
        clean_text = user_text.strip()
        if not clean_text:
            raise ValueError("El mensaje no puede estar vacío")

        run_id = str(uuid.uuid4())
        prompt = session.build_prompt(clean_text)
        self._start_run(run_id=run_id, session=session, mode="async")

        def on_retry(event: RetryEvent) -> None:
            self.trace_store.append(
                run_id=run_id,
                session_id=session.session_id,
                event="llm.retry",
                payload=asdict(event),
            )

        try:
            outcome = await generate_with_resilience(
                self.provider,
                prompt,
                system=system,
                timeout_seconds=self.settings.request_timeout_seconds,
                max_retries=self.settings.max_retries,
                base_delay_seconds=self.settings.retry_base_delay_seconds,
                on_retry=on_retry,
            )
            result = outcome.result
            session.add("user", clean_text)
            session.add("assistant", result.text)

            response = ChatResponse(
                run_id=run_id,
                session_id=session.session_id,
                text=result.text,
                attempts=outcome.attempts,
                provider=result.provider,
                model=result.model,
                latency_ms=result.latency_ms,
                input_tokens=result.input_tokens,
                output_tokens=result.output_tokens,
                estimated_cost_usd=self._estimate_cost(
                    result.input_tokens,
                    result.output_tokens,
                ),
            )
            self.trace_store.append(
                run_id=run_id,
                session_id=session.session_id,
                event="run.completed",
                payload=response.to_dict(),
            )
            return response
        except Exception as exc:
            self.trace_store.append(
                run_id=run_id,
                session_id=session.session_id,
                event="run.failed",
                payload={"error_type": type(exc).__name__, "message": str(exc)},
            )
            raise

    async def ask_stream(
        self,
        session: ConversationSession,
        user_text: str,
        *,
        system: str = "Sos un asistente útil, preciso y breve.",
    ) -> StreamChatResponse:
        clean_text = user_text.strip()
        if not clean_text:
            raise ValueError("El mensaje no puede estar vacío")

        run_id = str(uuid.uuid4())
        prompt = session.build_prompt(clean_text)
        self._start_run(run_id=run_id, session=session, mode="stream")

        started = time.perf_counter()
        chunks: list[str] = []
        first_chunk_ms: float | None = None

        try:
            async for chunk in self.provider.astream(prompt, system=system):
                chunks.append(chunk)
                if first_chunk_ms is None:
                    first_chunk_ms = round((time.perf_counter() - started) * 1000, 2)
        except Exception as exc:
            if chunks:
                error = PartialStreamError(
                    f"El stream falló después de {len(chunks)} fragmentos: {exc}"
                )
            else:
                error = exc
            self.trace_store.append(
                run_id=run_id,
                session_id=session.session_id,
                event="run.failed",
                payload={"error_type": type(error).__name__, "message": str(error)},
            )
            raise error

        text = "".join(chunks).strip()
        total_ms = round((time.perf_counter() - started) * 1000, 2)
        session.add("user", clean_text)
        session.add("assistant", text)

        response = StreamChatResponse(
            run_id=run_id,
            session_id=session.session_id,
            text=text,
            chunks=len(chunks),
            first_chunk_ms=first_chunk_ms or total_ms,
            total_ms=total_ms,
            provider="gemini" if type(self.provider).__name__ == "GeminiProvider" else "fake",
            model=getattr(self.provider, "model", "unknown"),
        )
        self.trace_store.append(
            run_id=run_id,
            session_id=session.session_id,
            event="run.completed",
            payload=response.to_dict(),
        )
        return response

Writing ai_agent_project/src/ai_agent_course/app.py


In [47]:
import ai_agent_course.app as app_module
import ai_agent_course.runtime as runtime_module
importlib.reload(runtime_module)
importlib.reload(app_module)

trace_path = ARTIFACTS_DIR / "class03_traces.jsonl"
if trace_path.exists():
    trace_path.unlink()

RUN_REAL_GEMINI_DEMO = True  # Cambiar a True únicamente si GEMINI_API_KEY está configurada y USE_REAL_GEMINI=1 en .env
if RUN_REAL_GEMINI_DEMO:
    # Provider de prueba
    provider = providers_module.GeminiProvider(settings.api_key_value(), settings.default_model)
    app_settings = settings

else:
    provider = providers_module.FakeProvider(
        model="fake:gemini-3.1-flash-lite",
        delay_seconds=0.03,
        failures=["rate_limit"],
    )
    app_settings = demo_settings
    

trace_store = runtime_module.JsonlTraceStore(trace_path)
# Si queremos usar el provider real de Gemini, descomentar la siguiente línea y comentar la anterior.
# chat_app = app_module.ConversationApp(lab_provider, demo_settings, trace_store)

chat_app = app_module.ConversationApp(provider, app_settings, trace_store)
lab_session = conversation_module.ConversationSession(
    app_settings.history_max_turns
)

first_response = await chat_app.ask(
    lab_session,
    "Estoy construyendo un cliente de LLM. ¿Qué responsabilidad debería tener el provider?",
)
streamed_response = await chat_app.ask_stream(
    lab_session,
    "¿Y qué debería quedar fuera de esa clase?",
)

print_json("Turno 1 · async con retry", first_response.to_dict())
print_json("Turno 2 · streaming", streamed_response.to_dict())
print_json("Resumen de sesión", {
    "session_id": lab_session.session_id,
    "messages": len(lab_session.messages),
    "runs": [first_response.run_id, streamed_response.run_id],
})


Turno 1 · async con retry
-------------------------
{
  "run_id": "ff34b339-9ff4-4ea2-aa76-d8b118340f2a",
  "session_id": 6,
  "text": "Para un cliente de LLM, el **provider** (la capa de abstracción que conecta con la API) debería tener estas responsabilidades clave:\n\n1.  **Normalización:** Traducir las respuestas de diferentes modelos (OpenAI, Anthropic, Ollama) a un formato único y consistente para tu aplicación.\n2.  **Gestión de la API:** Manejar la autenticación, los *headers* y la configuración de los *endpoints*.\n3.  **Manejo de errores:** Estandarizar las excepciones (ej. límites de tasa, errores de red) para que tu lógica de negocio no dependa de los errores específicos de cada proveedor.\n4.  **Abstracción de Streaming:** Proveer una interfaz uniforme para manejar los *chunks* de datos, independientemente de si el proveedor usa Server-Sent Events (SSE) o WebSockets.\n5.  **Transformación de parámetros:** Mapear parámetros comunes (como `temperature` o `max_tokens`) a los

In [48]:
events = trace_store.read_all()
trace_rows = []
for item in events:
    payload = item["payload"]
    detail = (
        payload.get("error_type")
        or (f"intentos={payload['attempts']}" if payload.get("attempts") else None)
        or (f"chunks={payload['chunks']}" if payload.get("chunks") else None)
        or payload.get("provider", "-")
    )
    trace_rows.append({
        "evento": item.get("event", "-"),
        "run_id": str(item.get("run_id", "-"))[:8],
        "session_id": str(item.get("session_id", "-"))[:8],
        "detalle": detail,
    })

print_table(trace_rows, ["evento", "run_id", "session_id", "detalle"])
print(f"\nEventos guardados: {len(events)} en {trace_path.relative_to(PROJECT_ROOT)}")

evento        | run_id   | session_id | detalle       
--------------+----------+------------+---------------
run.started   | ff34b339 | 6          | GeminiProvider
run.completed | ff34b339 | 6          | intentos=1    
run.started   | 893fb7c8 | 6          | GeminiProvider
run.completed | 893fb7c8 | 6          | chunks=13     

Eventos guardados: 4 en artifacts/class03_traces.jsonl


### Qué demuestra el caso integrador

- El primer turno sufre un rate limit simulado y termina bien en el segundo intento.
- El segundo turno reutiliza la misma sesión y entrega la respuesta por streaming.
- Cada turno tiene un `run_id` distinto.
- El historial se actualiza solo después de una respuesta exitosa.
- La interfaz recibe objetos claros, no `print()` internos.
- La traza permite reconstruir qué ocurrió sin guardar la API key.

## 12. Tests con dobles y mocks

Los tests verifican la lógica sin llamar a Gemini. El archivo queda visible en la notebook antes de ejecutarse.

In [49]:
%%writefile ai_agent_project/tests/test_class03.py
from __future__ import annotations

import asyncio
from unittest.mock import AsyncMock

from ai_agent_course.conversation import ConversationSession
from ai_agent_course.errors import ProviderConfigurationError
from ai_agent_course.providers import FakeProvider, GenerationResult
from ai_agent_course.resilience import generate_with_resilience


async def check_fake_provider_contract():
    result = await FakeProvider(delay_seconds=0).agenerate("hola")
    assert result.provider == "fake"
    assert result.text
    assert result.input_tokens > 0


async def check_retry_transient_error():
    provider = FakeProvider(delay_seconds=0, failures=["rate_limit"])
    outcome = await generate_with_resilience(
        provider,
        "mensaje válido",
        timeout_seconds=0.2,
        max_retries=1,
        base_delay_seconds=0,
    )
    assert outcome.attempts == 2
    assert len(outcome.retries) == 1


async def check_configuration_error_is_not_retried():
    provider = FakeProvider(delay_seconds=0, failures=["config"])
    try:
        await generate_with_resilience(
            provider,
            "mensaje válido",
            timeout_seconds=0.2,
            max_retries=3,
            base_delay_seconds=0,
        )
    except ProviderConfigurationError:
        return
    raise AssertionError("Se esperaba ProviderConfigurationError")


async def check_async_mock_called_once():
    provider = type("ProviderMock", (), {})()
    provider.agenerate = AsyncMock(
        return_value=GenerationResult(
            text="respuesta mock",
            model="mock-model",
            provider="mock",
            latency_ms=1,
            input_tokens=2,
            output_tokens=2,
        )
    )
    outcome = await generate_with_resilience(
        provider,
        "mensaje válido",
        timeout_seconds=0.2,
        max_retries=1,
        base_delay_seconds=0,
    )
    provider.agenerate.assert_awaited_once()
    assert outcome.result.text == "respuesta mock"


def test_fake_provider_contract():
    asyncio.run(check_fake_provider_contract())


def test_retry_transient_error():
    asyncio.run(check_retry_transient_error())


def test_configuration_error_is_not_retried():
    asyncio.run(check_configuration_error_is_not_retried())


def test_async_mock_called_once():
    asyncio.run(check_async_mock_called_once())


def test_history_window():
    session = ConversationSession(max_turns=1)
    session.add("user", "mensaje uno")
    session.add("assistant", "respuesta uno")
    session.add("user", "mensaje dos")
    session.add("assistant", "respuesta dos")
    assert len(session.messages) == 4
    assert len(session.recent_messages()) == 2

Writing ai_agent_project/tests/test_class03.py


In [50]:
async def run_notebook_tests() -> list[dict[str, str]]:
    checks = [
        ("provider_contract", providers_module.FakeProvider(delay_seconds=0).agenerate("hola")),
    ]

    rows: list[dict[str, str]] = []

    try:
        result = await checks[0][1]
        assert result.provider == "fake" and result.text
        rows.append({"test": "provider_contract", "status": "PASS"})
    except Exception as exc:
        rows.append({"test": "provider_contract", "status": "FAIL", "error": str(exc)})

    try:
        result = await resilience_module.generate_with_resilience(
            providers_module.FakeProvider(delay_seconds=0, failures=["rate_limit"]),
            "mensaje",
            max_retries=1,
            base_delay_seconds=0,
        )
        assert result.attempts == 2
        rows.append({"test": "retry_transient", "status": "PASS"})
    except Exception as exc:
        rows.append({"test": "retry_transient", "status": "FAIL", "error": str(exc)})

    try:
        mock_provider = type("ProviderMock", (), {})()
        mock_provider.agenerate = AsyncMock(
            return_value=providers_module.GenerationResult(
                text="respuesta mock",
                model="mock-model",
                provider="mock",
                latency_ms=1,
                input_tokens=2,
                output_tokens=2,
            )
        )
        result = await resilience_module.generate_with_resilience(
            mock_provider,
            "mensaje",
            max_retries=0,
        )
        mock_provider.agenerate.assert_awaited_once()
        assert result.result.text == "respuesta mock"
        rows.append({"test": "async_mock", "status": "PASS"})
    except Exception as exc:
        rows.append({"test": "async_mock", "status": "FAIL", "error": str(exc)})

    try:
        test_session = conversation_module.ConversationSession(max_turns=1)
        for role, text in [
            ("user", "uno"),
            ("assistant", "respuesta uno"),
            ("user", "dos"),
            ("assistant", "respuesta dos"),
        ]:
            test_session.add(role, text)
        assert len(test_session.recent_messages()) == 2
        rows.append({"test": "history_window", "status": "PASS"})
    except Exception as exc:
        rows.append({"test": "history_window", "status": "FAIL", "error": str(exc)})

    return rows


test_results = await run_notebook_tests()
print_table(test_results, ["test", "status"])
assert all(item["status"] == "PASS" for item in test_results)
print("\n✅ Todos los tests pasaron sin llamar a una API externa.")

test              | status
------------------+-------
provider_contract | PASS  
retry_transient   | PASS  
async_mock        | PASS  
history_window    | PASS  

✅ Todos los tests pasaron sin llamar a una API externa.


## 13. Desafío integrador

Extendé `ConversationApp` con una mejora:

1. separar `user_id` de `session_id`;
2. limitar mensajes a 2.000 caracteres;
3. implementar cancelación explícita;
4. persistir y recuperar sesiones;
5. crear un segundo provider y hacer routing por privacidad o costo.

La entrega debe incluir decisión, código, test, evidencia y trade-off.

In [51]:
student_decision = {
    "improvement": "",  # TODO
    "reason": "",       # TODO
    "test_added": "",   # TODO
    "tradeoff": "",     # TODO
}

print_json(
    "Registro del desafío",
    student_decision if any(student_decision.values()) else {
        "status": "pendiente",
        "instruction": "Completá student_decision después de implementar la mejora.",
    },
)


Registro del desafío
--------------------
{
  "status": "pendiente",
  "instruction": "Completá student_decision después de implementar la mejora."
}


## 14. Guardar evidencia y checkpoint

El reporte no contiene secretos ni el texto completo de la conversación. Resume decisiones, módulos creados y resultados verificables.

In [52]:
class03_report = {
    "class": 3,
    "mode": "real" if settings.use_real_gemini else "simulated",
    "model": settings.default_model,
    "configuration": settings.safe_dict(),
    "pedagogical_pattern": "understand_test_persist_import_verify",
    "modules_created": [
        "settings.py",
        "errors.py",
        "providers.py",
        "resilience.py",
        "conversation.py",
        "runtime.py",
        "app.py",
    ],
    "concurrency_experiment": {
        "sequential_ms": sequential_ms,
        "concurrent_ms": concurrent_ms,
        "max_concurrency": demo_settings.max_concurrency,
        "observed_speedup": round(sequential_ms / concurrent_ms, 2),
    },
    "retry_demo": {
        "attempts": prototype_outcome.attempts,
        "retries": [asdict(event) for event in prototype_outcome.retries],
    },
    "streaming_demo": {
        "chunks": len(stream_events),
        "first_chunk_ms": stream_events[0]["elapsed_ms"],
        "total_ms": stream_events[-1]["elapsed_ms"],
    },
    "conversation_demo": {
        "session_id": lab_session.session_id,
        "runs": 2,
        "messages": len(lab_session.messages),
        "trace_events": len(events),
    },
    "tests": test_results,
    "student_decision": student_decision,
}

report_path = ARTIFACTS_DIR / "class03_report.json"
report_path.write_text(
    json.dumps(class03_report, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("✅ Reporte guardado en:", report_path.relative_to(PROJECT_ROOT))

✅ Reporte guardado en: artifacts/class03_report.json


In [53]:
checks = {
    "project_created": PROJECT_ROOT.exists(),
    "requirements_created": (PROJECT_ROOT / "requirements.txt").exists(),
    "env_example_created": (PROJECT_ROOT / ".env.example").exists(),
    "settings_created": (SRC_DIR / "settings.py").exists(),
    "errors_created": (SRC_DIR / "errors.py").exists(),
    "providers_created": (SRC_DIR / "providers.py").exists(),
    "resilience_created": (SRC_DIR / "resilience.py").exists(),
    "conversation_created": (SRC_DIR / "conversation.py").exists(),
    "runtime_created": (SRC_DIR / "runtime.py").exists(),
    "app_created": (SRC_DIR / "app.py").exists(),
    "tests_created": (TESTS_DIR / "test_class03.py").exists(),
    "report_created": report_path.exists(),
    "tests_passed": all(item["status"] == "PASS" for item in test_results),
}

checkpoint_rows = [
    {"componente": name, "estado": "PASS" if passed else "FAIL"}
    for name, passed in checks.items()
]
print_table(checkpoint_rows, ["componente", "estado"])
assert all(checks.values()), "Hay componentes estructurales pendientes."
print("\n🎉 Checkpoint aprobado. El proyecto está listo para la Clase 4.")

componente           | estado
---------------------+-------
project_created      | PASS  
requirements_created | PASS  
env_example_created  | PASS  
settings_created     | PASS  
errors_created       | PASS  
providers_created    | PASS  
resilience_created   | PASS  
conversation_created | PASS  
runtime_created      | PASS  
app_created          | PASS  
tests_created        | PASS  
report_created       | PASS  
tests_passed         | PASS  

🎉 Checkpoint aprobado. El proyecto está listo para la Clase 4.


In [54]:
checkpoint_dir = ARTIFACTS_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint = {
    "class": 3,
    "canonical_model": "gemini-3.1-flash-lite",
    "increment": "providers, settings, resiliencia, sesiones y trazas",
    "default_mode": "simulated",
    "real_mode_controls": ["USE_REAL_GEMINI=1", "RUN_REAL_GEMINI_DEMO=True"],
    "next_class": 4,
}
checkpoint_path = checkpoint_dir / "class03_checkpoint.json"
checkpoint_path.write_text(
    json.dumps(checkpoint, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Checkpoint transversal:", checkpoint_path.relative_to(PROJECT_ROOT))

Checkpoint transversal: artifacts/checkpoints/class03_checkpoint.json


## Qué continúa en la Clase 4

La próxima notebook reutilizará `LLMProvider` para comparar un provider cloud con uno local, sin cambiar la capa de aplicación.

La decisión no será “qué modelo es mejor”, sino **qué capacidades necesita la aplicación y qué restricciones debe respetar**.

Conservá completa la carpeta `ai_agent_project/`.